In [1]:
import os 
import json
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import glob
from datetime import datetime, timedelta

In [2]:
#The dataset that loads a single tensor, used to calc min and max 
class LazyDataset(Dataset):
    #  data is stored in .pt files, each file contains a single tensor
    def __init__(self, data_files, transform=None):
        self.data_files = data_files  # List of paths to data files

    def __len__(self):
        return len(self.data_files)

    def __getitem__(self, idx):
        data=torch.load(self.data_files[idx]) # shape (1, 64, 64)
        #squeeze data at dimension 1
        data = data.unsqueeze(0) # shape (1, 64, 64)
        return {'data': data, 'source': self.data_files[idx]}

In [3]:
def compute_channel_min_max(dataset, output_file='channel_min_max.json'):
    loader = DataLoader(dataset, batch_size=2, shuffle=False, num_workers=2)

    channel_min = None
    channel_max = None

    for batch in loader:
        # batch: [B, C, H, W]
        if isinstance(batch, (tuple, list)):
            images = batch[0]
        else:
            images = batch['data']

        B, C, H, W = images.shape
        images = images.view(B, C, -1)  # [B, C, H*W]

        batch_min = images.min(dim=2)[0].min(dim=0)[0]  # [C]
        batch_max = images.max(dim=2)[0].max(dim=0)[0]  # [C]

        if channel_min is None:
            channel_min = batch_min
            channel_max = batch_max
        else:
            channel_min = torch.min(channel_min, batch_min)
            channel_max = torch.max(channel_max, batch_max)

    min_list = channel_min.tolist()
    max_list = channel_max.tolist()

    with open(output_file, 'w') as f:
        json.dump({'min': min_list, 'max': max_list}, f, indent=2)

    print(f"Saved channel-wise min/max to {output_file}")
    return min_list, max_list

In [4]:
#switch to your own data path!
input_dir='/scratch/nf33/cd3022/pyearth'
data_files=glob.glob(os.path.join(input_dir, '*.pt'))
len(data_files)

10

In [5]:
train_dataset = LazyDataset(data_files)
min_l, max_l = compute_channel_min_max(train_dataset, output_file=os.path.join(input_dir, 'min_max.json' ))

Saved channel-wise min/max to /scratch/nf33/cd3022/pyearth/min_max.json
